# 03 · E0 report and gate G1

**Runtime:** CPU works; GPU is faster for the head checks. **Time:** a few minutes.
Needs notebooks 01 and 02 to have finished.

Collects every E0 check into `WORKSPACE/reports/e0_report.md`:
- data checks from notebook 01, the floor baselines (random, longest answer, answer position)
- cache integrity and determinism from notebook 02
- all optical-map unit tests (`tests/test_optics.py`)
- head sanity: monotone training loss, permuted labels → 25 ± 3 %, Gaussian-noise inputs → 25 ± 3 %,
  optical map with β → 0 → constant features and chance accuracy

**G1 passes only if every HARD check passes.** WARN lines are reported but do not block
(informational, or properties of the official dataset we are not allowed to change).

In [ ]:
# ---- Setup (edit the two lines below once) ------------------------------------------
GITHUB_REPO = "YOUR_GITHUB_USERNAME/YOUR_REPO_NAME"              # repo that holds this code
WORKSPACE   = "/content/drive/MyDrive/optical_lm_workspace"      # splits, cache, runs, reports
# -------------------------------------------------------------------------------------
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["OPTLM_WORKSPACE"] = WORKSPACE
os.environ["HF_HOME"] = "/content/hf_cache"          # downloads stay on the fast local disk

# Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
REPO_DIR = "/content/" + GITHUB_REPO.split("/")[-1]
cmd = ["git", "clone", "-q", url, REPO_DIR] if not os.path.isdir(REPO_DIR) else ["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    msg = res.stderr.replace(token, "***") if token else res.stderr
    raise RuntimeError("git failed (check GITHUB_REPO / GH_TOKEN):\n" + msg)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("repo   :", REPO_DIR)
print("commit :", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())
print("dirty  :", bool(subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"]).strip()))
print("workspace:", WORKSPACE)

### All unit tests (data, heads, optics)

In [ ]:
!python -m pytest -q tests

### E0 report

In [ ]:
!python scripts/run_e0.py --config configs/e0.yaml

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(WORKSPACE, "reports", "e0_report.md"), encoding="utf-8").read()))